In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection       import train_test_split
from sklearn.linear_model         import LogisticRegression
from sklearn.metrics              import classification_report, confusion_matrix
import re
import numpy as np

train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

train.head()


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [2]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)  # linkleri sil
    text = re.sub(r'@\w+', '', text)     # @kullanıcıyı sil
    text = re.sub(r"#", "", text)  # hashtag işaretini sil ama kelimeyi bırak
    text = re.sub(r'\d+', '', text)           # SAYILARI sil
    text = re.sub(r'[^a-z\s]', '', text) # noktalama sil
    text = re.sub(r'\s+', ' ', text).strip()  # fazla boşlukları tek boşluğa indir
    return text

train['text_clean'] = train['text'].apply(clean_text)
test['text_clean'] = test['text'].apply(clean_text)

train[['text', 'text_clean']].head()




,text,text_clean
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...
3,"13,000 people receive #wildfires evacuation or...",people receive wildfires evacuation orders in ...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...


In [3]:

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3), stop_words='english')
X = vectorizer.fit_transform(train['text_clean'])

# Hedef değişken
y = train['target']

# Eğitim ve doğrulama için veriyi ayır
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
# Modeli oluştur ve eğit
model = LogisticRegression(C=0.7, max_iter=1000)
model.fit(X_train, y_train)

# Doğrulama verisi ile tahmin yap
y_pred = model.predict(X_val)

# Performansı göster
print(classification_report(y_val, y_pred))



              precision    recall  f1-score   support

           0       0.79      0.91      0.84       874
           1       0.84      0.67      0.74       649

    accuracy                           0.80      1523
   macro avg       0.81      0.79      0.79      1523
weighted avg       0.81      0.80      0.80      1523



In [5]:

# 1️⃣: Daha önce temizlediğimiz test verisini vektörleştiriyoruz
X_test = vectorizer.transform(test['text_clean'])

# 2️⃣: Modeli kullanarak tahmin yapıyoruz
predictions = model.predict(X_test)

# 3️⃣: Kaggle’ın beklediği formatta bir DataFrame oluşturuyoruz
submission = pd.DataFrame({
    'id': test['id'],        # test.csv'deki tweet ID'leri
    'target': predictions    # modelin tahmin ettiği sonuçlar (0 veya 1)
})

# 4️⃣: Dosyayı CSV olarak dışa aktarıyoruz
submission.to_csv("submission.csv", index=False)

# 5️⃣: Son kontrol
submission.head()


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


In [6]:
# Test verisine tahminleri ekle
test['prediction'] = predictions
test['prediction_label'] = test['prediction'].apply(lambda x: 'About the disaster' if x == 1 else 'Not about the disaster')


In [7]:

# Rastgele 5 tweet seçip tahminini yazdıralım
#sample = test.sample(5, random_state=42)
sample = test.sample(5)  # random_state yok, her seferinde farklı 5 satır seçer

for i, row in sample.iterrows():
    print(f"\nTweet: {row['text']}")
    print(f"Tahmin: {row['prediction_label']}")


Tweet: Tumblr collective meltdown. #sebastianstanisaliveandwell #civilwar ?????? http://t.co/WOc1TeVVMP
Tahmin: Not about the disaster

Tweet: Ahead of Print: A New Paradigm of Injuries From Terrorist Explosions as a Function of Explosion Setting Type.:... http://t.co/tqQc3yxBoR
Tahmin: Not about the disaster

Tweet: @wyattmccab you'd throw a can of Copenhagen wintergreen on the ground that would explode on your enemies and give them mouth cancer
Tahmin: Not about the disaster

Tweet: Remember to crowd around the baggage carousel like it's armageddon and the bags are the last remaining food items on earth you animals.
Tahmin: Not about the disaster

Tweet: the fires of hell for Julie #Extant writers....she better go down in flames when this is all over...#Extant
Tahmin: Not about the disaster


In [8]:
print(classification_report(y_val, y_pred, target_names=['Not about the disaster', 'About the disaster']))


                        precision    recall  f1-score   support

Not about the disaster       0.79      0.91      0.84       874
    About the disaster       0.84      0.67      0.74       649

              accuracy                           0.80      1523
             macro avg       0.81      0.79      0.79      1523
          weighted avg       0.81      0.80      0.80      1523



In [9]:
# Koefisiyenleri al ve en etkili kelimeleri sırala
feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

# En çok pozitif etki eden kelimeler (afet olduğunu söyleyen)
top_positive = np.argsort(coefs)[-10:]
print("🔺 Words that suggest disaster-related:")
for i in top_positive:
    print(f"{feature_names[i]} --> {coefs[i]:.4f}")

# En çok negatif etki eden kelimeler (afetle ilgili değil diyen)
top_negative = np.argsort(coefs)[:10]
print("\n🔻 Words that suggest it is not related to the disaster:")
for i in top_negative:
    print(f"{feature_names[i]} --> {coefs[i]:.4f}")
    


🔺 Words that suggest disaster-related:
train --> 1.7674
bombing --> 1.8224
wildfire --> 1.9066
buildings --> 2.0461
earthquake --> 2.0557
killed --> 2.1183
suicide --> 2.1256
california --> 2.2754
fires --> 2.3525
hiroshima --> 2.8495

🔻 Words that suggest it is not related to the disaster:
im --> -1.8609
new --> -1.8085
love --> -1.6421
screaming --> -1.5104
body --> -1.4345
ruin --> -1.4251
wrecked --> -1.4110
panicking --> -1.3815
just --> -1.3434
twister --> -1.3363
